# 07. So sánh & Phân tích 3 mô hình

Trả lời các câu hỏi:
- Model nào tốt nhất? Chênh lệch?
- Phân tích metrics, residuals
- Train vs Val: overfitting?
- Feature importance
- Error analysis

In [ ]:
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src.models import load_model
from src.evaluation import regression_metrics, residual_summary

sns.set_theme(style='whitegrid')
FEAT_DIR = Path('../data_features')
MODEL_DIR = Path('../models')
RES_DIR = Path('../results/metrics')
FIG_DIR = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
test = pd.read_parquet(FEAT_DIR / 'test.parquet')
TARGET = 'delay_hours'
y_test = test[TARGET]

results = {}
for name in ['random_forest', 'xgboost', 'lightgbm']:
    with open(RES_DIR / f'{name}.json') as f:
        results[name] = json.load(f)

summary = pd.DataFrame({
    name: r['test'] for name, r in results.items()
}).T
summary

## 1. So sánh metrics trên test

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2', 'MAPE']):
    summary[metric].plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_title(metric)
    ax.set_ylabel(metric)
    for i, v in enumerate(summary[metric]):
        ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'metrics_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Train vs Val (overfitting check)

In [ ]:
rows = []
for name, r in results.items():
    for split in ['train', 'val', 'test']:
        rows.append({'model': name, 'split': split, **r[split]})
df_split = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_split, x='model', y='MAE', hue='split', ax=ax)
ax.set_title('MAE: Train vs Val vs Test')
plt.tight_layout()
plt.savefig(FIG_DIR / 'overfit_check.png', dpi=120, bbox_inches='tight')
plt.show()
df_split

## 3. Residual analysis

In [ ]:
DROP = [TARGET, 'trip_id', 'load_id', 'driver_id', 'truck_id', 'trailer_id',
        'customer_id', 'route_id', 'dispatch_date']
feature_cols = [c for c in test.columns if c not in DROP
                 and pd.api.types.is_numeric_dtype(test[c])]
X_test = test[feature_cols]

preds = {}
for name in ['random_forest', 'xgboost', 'lightgbm']:
    m = load_model(MODEL_DIR / f'{name}.pkl')
    try:
        preds[name] = m.predict(X_test)
    except Exception as e:
        print(f'{name} predict needs full feature set:', e)

fig, axes = plt.subplots(1, len(preds), figsize=(5 * len(preds), 5))
for ax, (name, p) in zip(axes, preds.items()):
    res = y_test - p
    ax.scatter(p, res, alpha=0.3, s=8)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_title(f'{name} residuals')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Residual')
plt.tight_layout()
plt.savefig(FIG_DIR / 'residuals.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Feature importance comparison

In [ ]:
importances = {}
for name in ['random_forest', 'xgboost', 'lightgbm']:
    try:
        m = load_model(MODEL_DIR / f'{name}.pkl')
        importances[name] = pd.Series(m.feature_importances_, index=feature_cols).sort_values(ascending=False)
    except Exception:
        pass

if importances:
    fig, axes = plt.subplots(1, len(importances), figsize=(6 * len(importances), 8))
    if len(importances) == 1:
        axes = [axes]
    for ax, (name, imp) in zip(axes, importances.items()):
        imp.head(15).plot(kind='barh', ax=ax)
        ax.invert_yaxis()
        ax.set_title(f'{name} - top 15')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()

## 5. Error analysis: lỗi nằm ở đâu?

- Theo route
- Theo driver tenure bucket
- Theo truck age bucket
- Theo tháng / mùa

In [ ]:
best_name = summary['MAE'].idxmin()
print('Best model:', best_name)
best_pred = preds[best_name]
test_eval = test.copy()
test_eval['pred'] = best_pred
test_eval['abs_error'] = (y_test - best_pred).abs()

if 'route_id' in test_eval.columns:
    top_err = test_eval.groupby('route_id')['abs_error'].mean().sort_values(ascending=False).head(10)
    print('\nTop 10 routes có lỗi cao nhất:')
    print(top_err)

## 6. Kết luận

**Best model**: ...

**Lý do**: ...

**Insights**: ...